In [ ]:
import pandas as pd
import numpy as np
from workalendar.america import BrazilRioDeJaneiro

# ===============================================
# Leitura 
# ===============================================

dfbruto = pd.read_json("dados brutos/linha_913.jsonl", lines=True)

In [ ]:
# ===============================================
# Limpeza dos dados de GPS
# ===============================================

df_limpo = dfbruto.copy()

# Corrige vírgula decimal nas coordenadas
df_limpo["latitude"]   = df_limpo["latitude"].astype(str).str.replace(",", ".").astype(float)
df_limpo["longitude"]  = df_limpo["longitude"].astype(str).str.replace(",", ".").astype(float)
df_limpo["velocidade"] = pd.to_numeric(df_limpo["velocidade"], errors="coerce")

# Converte Unix -> datetime de Brasília
df_limpo["datahora"] = pd.to_datetime(df_limpo["datahora"].astype(float), unit="ms", utc=True) \
                        .dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)

# Mantém só as colunas que importam
df_limpo = df_limpo[["ordem", "latitude", "longitude", "velocidade", "datahora", "linha"]]

# Filtros de qualidade:
df_limpo = df_limpo[
    df_limpo["latitude"].between(-23.10, -22.74) &
    df_limpo["longitude"].between(-43.80, -43.10) &
    df_limpo["velocidade"].between(0, 100)
]

# Remove pings duplicados do mesmo ônibus
df_limpo = df_limpo.drop_duplicates(subset=["ordem", "datahora"])
df_limpo = df_limpo.sort_values(["ordem", "datahora"]).reset_index(drop=True)
df_limpo = df_limpo.drop(columns=["velocidade"])

print(f"GPS limpo: {len(df_limpo):,} registros | {df_limpo['datahora'].min()} -> {df_limpo['datahora'].max()}")
df_limpo.head(5)

GPS limpo: 2,005,795 registros | 2025-05-31 23:57:12 -> 2026-06-01 00:59:44


,ordem,latitude,longitude,datahora,linha
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913


In [ ]:
# ===============================================
# Leitura e limpeza dos dados climáticos
# ===============================================

df_limpo_clima = df_limpo.copy()

def ler_climatempo(path):
    clima = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8-sig", quotechar='"', na_values=["", " "])
    clima = clima.rename(columns={
        "Data"           : "data",
        "Hora (UTC)"     : "hora_utc",
        "Temp. Max. (C)" : "temp_max",
        "Temp. Min. (C)" : "temp_min",
        "Chuva (mm)"     : "chuva",
    })
    clima = clima[["data", "hora_utc", "temp_max", "temp_min", "chuva"]]
    clima["hora_utc"] = clima["hora_utc"].astype(str).str.zfill(4)
    clima["hora_utc"] = clima["hora_utc"].str[:2] + ":" + clima["hora_utc"].str[2:]

    clima["datahora_clima"] = pd.to_datetime(
        clima["data"] + " " + clima["hora_utc"],
        format="%d/%m/%Y %H:%M", utc=True
    ).dt.tz_convert("America/Sao_Paulo").dt.tz_localize(None)

    return clima.drop(columns=["data", "hora_utc"]).sort_values("datahora_clima")

clima = pd.concat([
    ler_climatempo("dados brutos/climatempo 01-06-25--01-01-26.csv"),
    ler_climatempo("dados brutos/climatempo 02-01-26--01-06-26.csv"),
], ignore_index=True).drop_duplicates("datahora_clima")

df_limpo_clima = pd.merge_asof(
    df_limpo.sort_values("datahora"),
    clima.rename(columns={"datahora_clima": "datahora"}),
    on="datahora",
    direction="nearest",
    tolerance=pd.Timedelta("1h")
)
df_limpo_clima = df_limpo_clima.sort_values(["ordem", "datahora"]).reset_index(drop=True)

df_limpo_clima.head(5)

,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913,38.5,34.7,0.0
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913,38.5,34.7,0.0


In [ ]:
# ===============================================
# Adicionamos features de tempo
# ===============================================

df_limpo_clima_feriados = df_limpo_clima.copy()

def adicionar_features_temporais(df, coluna="datahora"):
    dt = df[coluna]

    hora = dt.dt.hour
    dia_semana = dt.dt.dayofweek 

    df["hora_sin"] = np.sin(2 * np.pi * hora / 24)
    df["hora_cos"] = np.cos(2 * np.pi * hora / 24)
    df["dia_sin"]  = np.sin(2 * np.pi * dia_semana / 7)
    df["dia_cos"]  = np.cos(2 * np.pi * dia_semana / 7)

    is_fim_semana = dia_semana >= 5
    
    cal = BrazilRioDeJaneiro()
    feriados = set()
    for ano in dt.dt.year.unique():
        for data, _ in cal.holidays(int(ano)):
            feriados.add(data)

    is_feriado = dt.dt.date.apply(lambda d: d in feriados).astype(bool)

    # Diferencia fim de semana, feriado e dia útil
    condicoes = [is_feriado, is_fim_semana]
    escolhas = ['feriado', 'fim_semana']
    df['tipo_dia'] = np.select(condicoes, escolhas, default='dia_util')

    return df

df_limpo_clima_feriados = adicionar_features_temporais(df_limpo_clima_feriados)

In [ ]:
# ===============================================
# Vamos tirar linhas nulas
# ===============================================
df_final = df_limpo_clima_feriados.copy()

n_linhas_nulas = df_final.isnull().any(axis=1).sum()
pct_linhas_nulas = n_linhas_nulas / len(df_final) * 100
print(f'Linhas com pelo menos um NaN: {n_linhas_nulas} ({pct_linhas_nulas:.4f}%)')

# Tirando nulos
df_final = df_final.dropna().reset_index(drop=True)

df_final.head(5)

Linhas com pelo menos um NaN: 1886 (0.0940%)


,ordem,latitude,longitude,datahora,linha,temp_max,temp_min,chuva,hora_sin,hora_cos,dia_sin,dia_cos,tipo_dia
0,B10001,-22.81554,-43.18691,2025-12-16 13:08:21,913,38.4,36.2,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
1,B10001,-22.81552,-43.18688,2025-12-16 13:38:23,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
2,B10001,-22.81552,-43.18688,2025-12-16 13:43:01,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
3,B10001,-22.81552,-43.18688,2025-12-16 13:43:03,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util
4,B10001,-22.81554,-43.18689,2025-12-16 13:43:34,913,38.5,34.7,0.0,-0.258819,-0.965926,0.781831,0.62349,dia_util


In [6]:
df_final.to_csv("saida/linha_913_final.csv", index=False)
print("\nSalvo em saida/linha_913_final.csv")


Salvo em saida/linha_913_final.csv
